# Warehouse and conformance

The conformed dimensions, the facts, the validity rules and the conformance tests.

This notebook only presents results. Every number below was computed by the pipeline (`make all`) and is read from `results/` or from the warehouse through a read-only connection; nothing here trains a model or recomputes a reported number. Run it with `make notebooks`.

In [ ]:
profile = "full"

In [ ]:
import duckdb
import pandas as pd
from IPython.display import Image, display

from cigd.config import REPOSITORY_ROOT, load_config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config = load_config(profile)


def table(name: str) -> pd.DataFrame:
    """Read one result table written by the pipeline."""
    return pd.read_csv(config.tables_dir / f"{name}.csv")


def figure(name: str) -> None:
    """Show one figure written by the pipeline."""
    display(Image(filename=str(config.figures_dir / f"{name}.png"), width=900))


def warehouse_query(sql: str) -> pd.DataFrame:
    """Query the warehouse through a read-only connection."""
    with duckdb.connect(str(config.warehouse_path), read_only=True) as connection:
        return connection.execute(sql).df()

## Tables, grains and row counts

In [ ]:
table("warehouse_tables")

## Trip validity rules

`trips_failing_rule` counts every trip that fails a rule; `trips_assigned` counts the trips each rule removes when the rules are applied in order, so it adds up to the published total.

In [ ]:
table("warehouse_trip_validity")

## The geography hierarchy

Every member above zone level, with its ancestors. Newark Airport rolls up to New Jersey and PADD 1B; zones 264 and 265 roll up to Unknown.

In [ ]:
warehouse_query("""
SELECT geography_level, geography_code, geography_name,
       region_code, state_code, padd_code, country_code, is_nyserda_region
FROM dim_geography
WHERE geography_level <> 'zone'
ORDER BY geography_key
""")

## The week convention in dim_date

A NYSERDA price dated Monday w describes the week before it, so every day carries the label of its week.

In [ ]:
warehouse_query("""
SELECT full_date, day_name, week_start_date,
       nyserda_week_label_date, is_us_federal_holiday, us_federal_holiday_name
FROM dim_date
WHERE full_date BETWEEN DATE '2024-12-23' AND DATE '2025-01-06'
ORDER BY full_date
""")

## Conformance and other tests

The test stage's outcome counts, from `results/run_log.json`.

In [ ]:
import json

run_log = json.loads(config.run_log_path.read_text(encoding="utf-8"))
pd.Series(run_log["stages"]["test"]["summary"])